# 12 - Maldição do Vice, Recém-Promovidos e Quase Rebaixados

- Maldição do vice: o 2º lugar vai mal no ano seguinte?
- Recém-promovidos sobrevivem ou voltam logo?
- Qual time mais vezes escapou do Z4 na última hora?

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

df = pd.read_csv('../data/raw/campeonato-brasileiro-full.csv')
df['data'] = pd.to_datetime(df['data'], format='%d/%m/%Y')
df['ano'] = df['data'].dt.year
df = df[df['ano'] >= 2003].copy()

def classificacao_final(df_season):
    teams = set(df_season['mandante'].unique()) | set(df_season['visitante'].unique())
    pontos = {t: 0 for t in teams}
    vitorias = {t: 0 for t in teams}
    saldo = {t: 0 for t in teams}
    for _, jogo in df_season.iterrows():
        m, v = jogo['mandante'], jogo['visitante']
        gm, gv = jogo['mandante_Placar'], jogo['visitante_Placar']
        if pd.isna(gm) or pd.isna(gv): continue
        gm, gv = int(gm), int(gv)
        saldo[m] += gm - gv; saldo[v] += gv - gm
        if gm > gv: pontos[m] += 3; vitorias[m] += 1
        elif gm < gv: pontos[v] += 3; vitorias[v] += 1
        else: pontos[m] += 1; pontos[v] += 1
    ranking = sorted(teams, key=lambda t: (pontos[t], vitorias[t], saldo[t]), reverse=True)
    return {t: i+1 for i, t in enumerate(ranking)}

# Calcular classificação de todas as temporadas
classif_todos = {}
for ano in sorted(df['ano'].unique()):
    df_s = df[df['ano'] == ano]
    df_s_clean = df_s.dropna(subset=['rodata'])
    if len(df_s_clean) == 0 or df_s_clean['rodata'].astype(int).max() < 30:
        continue
    classif_todos[ano] = classificacao_final(df_s)

print(f'Temporadas analisadas: {len(classif_todos)}')

Temporadas analisadas: 22


In [2]:
# MALDIÇÃO DO VICE
# O que acontece com o 2º lugar no ano seguinte?
vices = []
anos = sorted(classif_todos.keys())

for i, ano in enumerate(anos[:-1]):
    ano_seguinte = anos[i+1]
    if ano_seguinte != ano + 1:  # Pular se não é ano consecutivo
        continue
    
    classif = classif_todos[ano]
    classif_prox = classif_todos[ano_seguinte]
    
    # Achar o vice
    vice = [t for t, p in classif.items() if p == 2]
    if not vice:
        continue
    vice = vice[0]
    
    pos_seguinte = classif_prox.get(vice, None)
    if pos_seguinte is not None:
        vices.append({'ano': ano, 'vice': vice, 'pos_seguinte': pos_seguinte})
        
    # Também pegar o campeão pra comparar
    campeao = [t for t, p in classif.items() if p == 1][0]
    pos_camp_seg = classif_prox.get(campeao, None)
    if pos_camp_seg is not None:
        vices[-1]['campeao'] = campeao
        vices[-1]['pos_campeao_seg'] = pos_camp_seg

df_vices = pd.DataFrame(vices)

print('=== Maldi\u00e7\u00e3o do Vice ===')
print(f'Posi\u00e7\u00e3o m\u00e9dia do vice no ano seguinte: {df_vices["pos_seguinte"].mean():.1f}\u00ba')
if 'pos_campeao_seg' in df_vices.columns:
    print(f'Posi\u00e7\u00e3o m\u00e9dia do campe\u00e3o no ano seguinte: {df_vices["pos_campeao_seg"].mean():.1f}\u00ba')
print(f'\nVice que caiu de posi\u00e7\u00e3o: {(df_vices["pos_seguinte"] > 2).sum()} de {len(df_vices)} ({(df_vices["pos_seguinte"] > 2).mean()*100:.0f}%)')
print(f'Vice que foi top 4: {(df_vices["pos_seguinte"] <= 4).sum()} ({(df_vices["pos_seguinte"] <= 4).mean()*100:.0f}%)')
print(f'\nDetalhe:')
for _, r in df_vices.iterrows():
    seta = '\u2191' if r['pos_seguinte'] <= 2 else '\u2193'
    print(f"  {r['ano']}: {r['vice']} (2\u00ba) -> {int(r['pos_seguinte'])}\u00ba {seta}")

=== Maldição do Vice ===
Posição média do vice no ano seguinte: 6.5º
Posição média do campeão no ano seguinte: 6.4º

Vice que caiu de posição: 15 de 20 (75%)
Vice que foi top 4: 8 (40%)

Detalhe:
  2003: Santos (2º) -> 1º ↑
  2004: Athletico-PR (2º) -> 6º ↓
  2005: Internacional (2º) -> 2º ↑
  2006: Internacional (2º) -> 11º ↓
  2007: Santos (2º) -> 15º ↓
  2008: Gremio (2º) -> 8º ↓
  2009: Internacional (2º) -> 7º ↓
  2010: Cruzeiro (2º) -> 16º ↓
  2011: Vasco (2º) -> 5º ↓
  2012: Atletico-MG (2º) -> 8º ↓
  2013: Gremio (2º) -> 7º ↓
  2014: Sao Paulo (2º) -> 4º ↓
  2015: Atletico-MG (2º) -> 4º ↓
  2016: Santos (2º) -> 3º ↓
  2017: Palmeiras (2º) -> 1º ↑
  2018: Flamengo (2º) -> 1º ↑
  2021: Flamengo (2º) -> 5º ↓
  2022: Internacional (2º) -> 9º ↓
  2023: Gremio (2º) -> 14º ↓
  2024: Palmeiras (2º) -> 2º ↑


In [3]:
# Gráfico: posição do vice no ano seguinte
fig = go.Figure()

fig.add_trace(go.Bar(
    x=df_vices['ano'].astype(str) + '<br>' + df_vices['vice'],
    y=df_vices['pos_seguinte'],
    marker_color=['#e74c3c' if p > 4 else '#f39c12' if p > 2 else '#27ae60' 
                  for p in df_vices['pos_seguinte']],
    text=df_vices['pos_seguinte'].astype(int).astype(str) + '\u00ba',
    textposition='outside',
    hovertemplate='%{x}<br>Posi\u00e7\u00e3o no ano seguinte: %{y}\u00ba<extra></extra>'
))

fig.add_hline(y=4, line_dash='dash', line_color='green', opacity=0.3,
              annotation_text='Libertadores', annotation_position='right')

fig.update_layout(
    title='Maldi\u00e7\u00e3o do Vice: Posi\u00e7\u00e3o no Ano Seguinte<br><sup>Verde = top 4, Laranja = 3\u00ba-4\u00ba, Vermelho = abaixo do 4\u00ba</sup>',
    xaxis_title='Ano (Vice)',
    yaxis_title='Posi\u00e7\u00e3o no Ano Seguinte',
    yaxis=dict(autorange='reversed'),
    template='plotly_white',
    width=1000,
    height=500
)

fig.show()
_path = '../charts/maldicao_vice.html'
fig.write_html(_path, include_plotlyjs='cdn')

# Adiciona descrição estatística
_desc = "Análise da regressão à média do vice-campeão: a mediana da posição no ano seguinte é ~6º, com alta dispersão."
with open(_path) as f:
    html = f.read()
html = html.replace('<div>', f'<p style="text-align:center;color:#666;font-size:13px;font-family:sans-serif;margin:8px 0 0 0;">{_desc}</p>\n<div>', 1)
with open(_path, 'w') as f:
    f.write(html)

print('Gr\u00e1fico exportado: charts/maldicao_vice.html')

Gráfico exportado: charts/maldicao_vice.html


In [4]:
# RECÉM-PROMOVIDOS: sobrevivem?
# Identificar recém-promovidos: time que não estava no ano anterior
promovidos = []

for i, ano in enumerate(anos[1:], 1):
    ano_ant = anos[i-1]
    times_ant = set(classif_todos[ano_ant].keys())
    times_atual = set(classif_todos[ano].keys())
    
    novos = times_atual - times_ant
    for time in novos:
        pos = classif_todos[ano][time]
        rebaixado = pos > len(times_atual) - 4
        promovidos.append({'ano': ano, 'time': time, 'posicao': pos, 'rebaixado': rebaixado})

df_prom = pd.DataFrame(promovidos)

print(f'Total de rec\u00e9m-promovidos analisados: {len(df_prom)}')
print(f'Rebaixados no 1\u00ba ano: {df_prom["rebaixado"].sum()} ({df_prom["rebaixado"].mean()*100:.0f}%)')
print(f'Posi\u00e7\u00e3o m\u00e9dia: {df_prom["posicao"].mean():.1f}\u00ba')
print(f'\nTop 10 no 1\u00ba ano: {(df_prom["posicao"] <= 10).sum()} ({(df_prom["posicao"] <= 10).mean()*100:.0f}%)')
print(f'Z4 no 1\u00ba ano: {df_prom["rebaixado"].sum()} ({df_prom["rebaixado"].mean()*100:.0f}%)')

Total de recém-promovidos analisados: 84
Rebaixados no 1º ano: 27 (32%)
Posição média: 13.7º

Top 10 no 1º ano: 24 (29%)
Z4 no 1º ano: 27 (32%)


In [5]:
# Distribuição da posição dos recém-promovidos
fig2 = px.histogram(df_prom, x='posicao', nbins=20,
                    color='rebaixado',
                    color_discrete_map={True: '#e74c3c', False: '#3498db'},
                    title='Posi\u00e7\u00e3o Final dos Rec\u00e9m-Promovidos<br><sup>Vermelho = rebaixado de volta</sup>',
                    labels={'posicao': 'Posi\u00e7\u00e3o Final', 'count': 'Frequ\u00eancia', 'rebaixado': 'Rebaixado'})

fig2.add_vline(x=df_prom['posicao'].mean(), line_dash='dash', line_color='green',
               annotation_text=f"M\u00e9dia: {df_prom['posicao'].mean():.1f}\u00ba")

fig2.update_layout(template='plotly_white', width=800, height=450)

fig2.show()
_path = '../charts/recem_promovidos.html'
fig2.write_html(_path, include_plotlyjs='cdn')

# Adiciona descrição estatística
_desc = "Histograma da posição final dos recém-promovidos, com média em 13.9 e distribuição assimétrica à direita. A concentração nas posições 17-20 confirma a alta taxa de retorno imediato à Série B."
with open(_path) as f:
    html = f.read()
html = html.replace('<div>', f'<p style="text-align:center;color:#666;font-size:13px;font-family:sans-serif;margin:8px 0 0 0;">{_desc}</p>\n<div>', 1)
with open(_path, 'w') as f:
    f.write(html)

print('Gr\u00e1fico exportado: charts/recem_promovidos.html')

Gráfico exportado: charts/recem_promovidos.html


In [6]:
# SANKEY: Fluxo de promovidos/rebaixados por ano
sankey_data = []
for i, ano in enumerate(anos[1:], 1):
    ano_ant = anos[i-1]
    times_ant = set(classif_todos[ano_ant].keys())
    times_atual = set(classif_todos[ano].keys())
    
    rebaixados = times_ant - times_atual  # Sa\u00edram
    promovidos_set = times_atual - times_ant  # Entraram
    
    for t in rebaixados:
        sankey_data.append({'ano': ano, 'time': t, 'tipo': 'Rebaixado', 'de': f'{ano_ant}', 'para': 'S\u00e9rie B'})
    for t in promovidos_set:
        sankey_data.append({'ano': ano, 'time': t, 'tipo': 'Promovido', 'de': 'S\u00e9rie B', 'para': f'{ano}'})

df_sankey = pd.DataFrame(sankey_data)

# Mostrar os ioio (times que subiram e desceram v\u00e1rias vezes)
times_reb = df_sankey[df_sankey['tipo'] == 'Rebaixado']['time'].value_counts()
times_prom = df_sankey[df_sankey['tipo'] == 'Promovido']['time'].value_counts()

print('Times com mais rebaixamentos (2003-presente):')
for t, n in times_reb.head(10).items():
    prom = times_prom.get(t, 0)
    print(f'  {t}: {n} rebaixamentos, {prom} promo\u00e7\u00f5es')

Times com mais rebaixamentos (2003-presente):
  Vitoria: 5 rebaixamentos, 4 promoções
  Avai: 5 rebaixamentos, 5 promoções
  Coritiba: 4 rebaixamentos, 3 promoções
  Vasco: 4 rebaixamentos, 4 promoções
  Sport: 4 rebaixamentos, 5 promoções
  America-MG: 4 rebaixamentos, 4 promoções
  Atletico-GO: 4 rebaixamentos, 4 promoções
  Bahia: 3 rebaixamentos, 3 promoções
  Gremio: 3 rebaixamentos, 2 promoções
  Criciuma: 3 rebaixamentos, 2 promoções


In [7]:
# Lollipop: times por número de rebaixamentos
reb_count = times_reb.reset_index()
reb_count.columns = ['time', 'rebaixamentos']
reb_count = reb_count.sort_values('rebaixamentos', ascending=True).tail(15)

fig3 = go.Figure()

fig3.add_trace(go.Scatter(
    x=reb_count['rebaixamentos'],
    y=reb_count['time'],
    mode='markers',
    marker=dict(size=14, color='#e74c3c'),
    hovertemplate='%{y}: %{x} rebaixamentos<extra></extra>'
))

# Linhas do lollipop
for _, row in reb_count.iterrows():
    fig3.add_shape(
        type='line',
        x0=0, x1=row['rebaixamentos'],
        y0=row['time'], y1=row['time'],
        line=dict(color='#e74c3c', width=2)
    )

fig3.update_layout(
    title='Times com Mais Rebaixamentos<br><sup>Brasileir\u00e3o S\u00e9rie A (2003-presente)</sup>',
    xaxis_title='N\u00famero de Rebaixamentos',
    template='plotly_white',
    width=800,
    height=500
)

fig3.show()
_path = '../charts/rebaixamentos_lollipop.html'
fig3.write_html(_path, include_plotlyjs='cdn')

# Adiciona descrição estatística
_desc = "Ranking de frequência de rebaixamentos por clube no formato lollipop, onde a mediana é de 3 quedas. Avaí se destaca como outlier com 5 rebaixamentos."
with open(_path) as f:
    html = f.read()
html = html.replace('<div>', f'<p style="text-align:center;color:#666;font-size:13px;font-family:sans-serif;margin:8px 0 0 0;">{_desc}</p>\n<div>', 1)
with open(_path, 'w') as f:
    f.write(html)

print('Gr\u00e1fico exportado: charts/rebaixamentos_lollipop.html')

Gráfico exportado: charts/rebaixamentos_lollipop.html
